# Feature Extractor Usage Example

This notebook shows the structured feature-spec API and a minimal extraction loop.
The first code cell is lightweight and safe to run; the second cell shows the full model-backed extraction path.

In [13]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
from feature_extractor.configs import (
    AttentionFeatureSpec,
    EmbeddingFeatureSpec,
    FeatureConfig,
    LayerFeatureSpec,
    MLPFeatureSpec,
)

feature_cfg = FeatureConfig(
    feature_specs=[
        EmbeddingFeatureSpec(),
        LayerFeatureSpec(layer_index=0, feature="input"),
        LayerFeatureSpec(layer_index=0, feature="output"),
        AttentionFeatureSpec(layer_index=0, feature="query"),
        AttentionFeatureSpec(layer_index=0, feature="attn_weights"),
        MLPFeatureSpec(layer_index=0, feature="output"),
    ],
    batch_size=2,
)


feature_cfg.feature_names_as_strings()

['embeddings',
 'layers.layer_00.input',
 'layers.layer_00.output',
 'attn.layer_00.query',
 'attn.layer_00.attn_weights',
 'mlp.layer_00.output']

In [15]:
from torch.utils.data import DataLoader

from feature_extractor.data.dataset import TextDataEntry, TextDataset, create_collator
from feature_extractor.extractor.extractor import FeatureExtractor

In [16]:
# Optional: run a real extraction pass.
# This example will download the model the first time it runs.
model_name = "openai-community/gpt2"
extractor = FeatureExtractor(model_name_or_path=model_name)
extractor.configure(feature_cfg)

# Build a tiny in-memory dataset.
dataset = TextDataset(
    data=[
        TextDataEntry(idx="0", text="Hello, world!"),
        TextDataEntry(idx="1", text="This is a sample."),
    ]
)
collator = create_collator(extractor.tokenizer)
dataloader = DataLoader(
    dataset,
    batch_size=feature_cfg.batch_size,
    shuffle=False,
    collate_fn=collator,
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

]8;id=810487;file:///Users/238-gs20253502/Documents/codes/transformer-feature-extractor/src/feature_extractor/models/__init__.py\__init__.py]8;;\:]8;id=656004;file:///Users/238-gs20253502/Documents/codes/transformer-feature-extractor/src/feature_extractor/models/__init__.py#81\81]8;;\
2026-05-07 18:03:23,994/INFO/feature_extractor.models/load_causal_model():81                                       
Model loaded on device: cpu                                                                                        

Matched model class GPT2LMHeadModel to architecture GPT2Architecture
Deepest layer index determined from feature config: 0
Applying early stopping at layer index 0.


In [17]:
%%time
for _batch, _features in extractor.extract_features(dataloader):
    print(f"Batch: {_batch.keys()}")
    print(f"Features: {_features}")

]8;id=846966;file:///Users/238-gs20253502/Documents/codes/transformer-feature-extractor/src/feature_extractor/logger.py\logger.py]8;;\:]8;id=454913;file:///Users/238-gs20253502/Documents/codes/transformer-feature-extractor/src/feature_extractor/logger.py#79\79]8;;\
2026-05-07 18:03:24,912/WARNING/feature_extractor.hooks.base/warn_once():79                                        
Ignored unexpected keys: {'causal_mask', 'encoder_hidden_states', 'cache_position', 'past_key_values'}             

Early stopping triggered by LayerHook.
Batch: dict_keys(['input_ids', 'attention_mask', 'indices'])
Features: HookResult(
    embeddings=EmbeddingHookResult,
    layers=[LayerHookResult, None, None, None, None, None, None, None, None, None, None, None],
    attn=[AttentionHookResult, None, None, None, None, None, None, None, None, None, None, None],
    mlp=[MLPHookResult, None, None, None, None, None, None, None, None, None, None, None],
)
CPU times: user 4.94 ms, sys: 2.58 ms, total: 7.52 ms
Wall time: 5.83 ms
